# Forecast metrics — benchmarking SMARD's day-ahead forecasts

Implements [`.claude/specs/04-forecast-metrics.md`](../../.claude/specs/04-forecast-metrics.md).

SMARD publishes a day-ahead forecast of residual load (`fc_residual_load`) and of its two parts
(`fc_grid_load`, `fc_gen_wind_solar`). That public forecast is the **baseline our own model has to
beat**. This notebook measures how good it is.

## What this notebook produces

- The **hourly error series** `error = forecast − actual` for three pairs:
  - residual load: `fc_residual_load` ↔ `residual_load`
  - grid load: `fc_grid_load` ↔ `grid_load`
  - wind + solar: `fc_gen_wind_solar` ↔ `renewables`
- **MAE, RMSE, bias and `hour_count`** per pair, plus nMAE and a capacity-normalised generation error
- slices by **time** (month, hour of day, season, year), by **residual-load level**, and the error of the **day maximum / minimum**
- a **cascade** of error tables (full record → trailing 365 days → year → month → day), all from one `window_metrics` helper
- three exported files in `data/metrics/`:
  - `smard_forecast_errors_hourly.csv`: the source of truth for re-scoring SMARD on any window
  - `smard_forecast_errors_daily.csv`
  - `smard_benchmark_metrics.csv`

## Conventions

- **Sign:** `error = forecast − actual`. **Positive error / positive bias = over-forecast**, negative = under-forecast.
- **Every metric is an average over a window of hours** and is always shown with its `hour_count`. The headline numbers are provisional: the modelling spec re-scores SMARD on its own test window.
- **Units:** hourly readings and hourly errors in `MWh`; normalised errors in `%`; timing offsets in hours (`h`); installed capacity in `MW`. An hourly `MWh` reading equals the mean power over that hour, so `err_renewables / cap_wind_solar` reads as a share of installed capacity.
- `time_series` is the shared hourly frame, holding exactly `SERIES + DERIVED`. The errors live in a separate `errors` frame.
- No literal calendar year or date appears in code. Everything year-dependent derives from `YEARS` or from the data.
- **Durations, never row counts.**

## Not in this notebook

- anything using the risk-label files (error on risk hours, flag agreement): parked in [04.3](../../.claude/specs/04.3-risk-label-link.md)
- naive baselines and skill scores: parked in [04.1](../../.claude/specs/04.1-naive-baseline.md)
- ISO-week level, zoom demos, tolerance share: parked in [04.2](../../.claude/specs/04.2-streamlit-views.md)
- MAPE, any model of our own, the train/test split, MLflow, reBAP

## 1 Setup

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) §1. Inherited, not re-derived:

- the data-directory resolver (walks **upward** from the working directory, so the notebook runs from any folder in the repo)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping
- `_complete_periods`, `period_mean`, `period_energy`, `style_timeseries`, `seasonal_plot`
- the duration constants and the day-completeness rule from `risk-definition.ipynb` §3.1

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"Data directory: {DATA}")

### 1.1 Helpers

Unchanged from `team-EDA.ipynb`:

- `_complete_periods` drops calendar periods the data does not fully cover. A plain `.resample()` produces fake edge dips.
- `style_timeseries` requires an explicit `ylabel`, so no plot can ship without stating its unit.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    A period counts only if it starts not earlier than the first observation and ends not later
    than the last observation's closing edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (
        periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts.
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view.
        Deliberately not ``mwh_per_day / 24``: a month containing the spring DST switch holds
        743 hours, not 744.
    ``hours``, ``days``
        the two denominators.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over tens of thousands of rows.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

### 1.2 Load and prepare

- The dtype assertion guards against the German Excel-CSV conversion silently leaving a column as text.
- Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar"
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.3 Engineered columns

- `YEARS` is computed from the loaded data and is the only permitted source of year information.
- `spans_gap` marks the row *following* a gap: at each spring DST switch the local hour **02:00 does not exist**.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.4 Rule constants, expressed as durations

Inherited from `risk-definition.ipynb` §3.1. Every rule about a length of time is written as a
**duration** and converted to an observation count from the **measured** resolution:

- the trailing window is **365 days**, not 8,760 rows
- a day is **complete** if it carries at least **23/24** of its expected observations. This accepts the 23-hour spring-DST day and rejects materially short days.

A switch to SMARD's quarter-hour data would therefore change the observation counts but none of the definitions.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

WINDOW = pd.Timedelta(days=365)      # trailing window: headline table and rolling lines
DAY_COMPLETENESS = 23 / 24           # accepts the spring-DST day, rejects materially short days

EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

# One row per local calendar date — the same day boundary DERIVED["date"] uses, not a rolling 24h.
DAY = pd.Series(time_series.index.normalize(), index=time_series.index)
DAYS = pd.DatetimeIndex(sorted(DAY.unique()))

print(f"resolution        : {RESOLUTION}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"trailing window   : {WINDOW.days} days")
print(f"day completeness  : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")
print(f"calendar days     : {len(DAYS):,}  ({DAYS.min():%Y-%m-%d} .. {DAYS.max():%Y-%m-%d})")

### 1.5 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. The closing self-check (§9) re-runs these invariants plus a comparison
against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

**What we are working with**

- **One hourly record, 67,336 rows (2019-01-01 00:00 → 2026-09-06 23:00), 2,806 calendar days.**
  - The final year is **partial** (it ends in September). Every year-on-year statement below therefore comes from full 365-day windows, never from a partial year against a full one.
  - The extent is a snapshot and moves with every re-fetch. Nothing below depends on these numbers being fixed.

- **Every forecast/actual pair already sits side by side in the first row.**
  - At 2019-01-01 00:00 SMARD forecast `fc_residual_load` = 19,264.5 MWh against an actual `residual_load` of 19,825.5 MWh. That is an error of **−561 MWh**: an under-forecast under this notebook's sign convention.
  - §2 turns this into an error series for all three pairs and every hour.

---

## 2 Errors and their decomposition

The errors live in a **separate `errors` frame** on `time_series.index`, not as new columns on
`time_series`, so `time_series` keeps exactly `SERIES + DERIVED`.

Per pair the frame holds the actual, the forecast and `err_{actual} = forecast − actual`. These are
the same pair columns the hourly export writes in §8:

| Pair | Forecast | Actual | Error column |
|---|---|---|---|
| residual load | `fc_residual_load` | `residual_load` | `err_residual_load` |
| grid load | `fc_grid_load` | `grid_load` | `err_grid_load` |
| wind + solar | `fc_gen_wind_solar` | `renewables` | `err_renewables` |

It also carries the installed wind + solar capacity
`cap_wind_solar = cap_wind_off + cap_wind_on + cap_solar` (MW), so `window_metrics` (§3) can compute
nMAE and the capacity-normalised generation error from this one frame.

**Sign convention: positive error / positive bias = over-forecast, negative = under-forecast.**

In [ ]:
# One entry per forecast/actual pair, keyed by the actual column. Every later section loops over this.
PAIRS = {
    "residual_load": {"forecast": "fc_residual_load", "label": "Residual load", "color": "#1C1C1C"},
    "grid_load": {"forecast": "fc_grid_load", "label": "Grid load", "color": "#2C6EBA"},
    "renewables": {"forecast": "fc_gen_wind_solar", "label": "Wind + solar", "color": "#2E8B57"},
}
ERR = {actual: f"err_{actual}" for actual in PAIRS}

errors = pd.DataFrame(index=time_series.index)
for actual, spec in PAIRS.items():
    errors[actual] = time_series[actual]
    errors[spec["forecast"]] = time_series[spec["forecast"]]
    # NaN wherever either side is missing, so the pairwise-complete rule (2.2) is built in.
    errors[ERR[actual]] = time_series[spec["forecast"]] - time_series[actual]

errors["cap_wind_solar"] = time_series[["cap_wind_off", "cap_wind_on", "cap_solar"]].sum(axis=1)

assert errors.index.equals(time_series.index)
assert list(time_series.columns) == SERIES + DERIVED, "time_series must not gain error columns"

print(f"errors: {errors.shape[0]:,} rows x {errors.shape[1]} columns")
print(f"columns: {list(errors.columns)}")
errors.head(3)

### 2.1 The forecast-side identity and the error decomposition

`risk-definition.ipynb` §1.7 showed that the actual residual load is load minus wind and solar:
`residual_load = grid_load − renewables`. If SMARD builds its forecast the same way,

$$\text{fc\_residual\_load} = \text{fc\_grid\_load} - \text{fc\_gen\_wind\_solar},$$

then subtracting the two identities gives an exact decomposition of the error:

$$\text{err\_residual\_load} = \text{err\_grid\_load} - \text{err\_renewables}.$$

Both identities are checked below. The decomposition can deviate by at most the sum of the two
identity deviations, and that bound is the tolerance the closing self-check (§9) uses.

In [ ]:
def identity_deviation(total, load, generation):
    """|total − (load − generation)| on the hours where all three exist."""
    return (total - (load - generation)).abs().dropna()


fc_dev = identity_deviation(
    time_series["fc_residual_load"], time_series["fc_grid_load"], time_series["fc_gen_wind_solar"]
)
act_dev = identity_deviation(
    time_series["residual_load"], time_series["grid_load"], time_series["renewables"]
)
decomp_dev = identity_deviation(
    errors["err_residual_load"], errors["err_grid_load"], errors["err_renewables"]
)

# Algebraic bound: the decomposition deviation is the forecast-side minus the actual-side deviation.
DECOMP_TOL = fc_dev.max() + act_dev.max()

for name, dev in [("forecast identity", fc_dev), ("actual identity", act_dev),
                  ("error decomposition", decomp_dev)]:
    over_1 = dev > 1
    line = (f"{name:20s}: {len(dev):,} hours, exact {(dev == 0).mean():6.2%}, "
            f"<= 1 MWh {(dev <= 1).mean():7.3%}, max {dev.max():,.2f} MWh")
    if over_1.any():
        line += (f"  |  {int(over_1.sum())} hours > 1 MWh, "
                 f"{dev[over_1].index.min():%Y-%m-%d} .. {dev[over_1].index.max():%Y-%m-%d}")
    print(line)

print(f"\ndecomposition tolerance DECOMP_TOL = {DECOMP_TOL:,.2f} MWh "
      f"(max forecast-side + max actual-side deviation)")
assert decomp_dev.max() <= DECOMP_TOL + 1e-9

# A worked example from the data: the hour with the largest residual-load error.
worst = errors["err_residual_load"].abs().idxmax()
e = errors.loc[worst]
print(f"\nlargest |err_residual_load| at {worst}:")
print(f"  err_grid_load     {e['err_grid_load']:>10,.0f} MWh")
print(f"  err_renewables    {e['err_renewables']:>10,.0f} MWh")
print(f"  err_residual_load {e['err_residual_load']:>10,.0f} MWh "
      f"(= {e['err_grid_load']:,.0f} − ({e['err_renewables']:,.0f}))")

**Both identities hold, so every residual-load error splits exactly into a load part and a generation part.**

- **SMARD builds its residual-load forecast as load minus wind + solar.**
  - The forecast identity holds in all 67,312 hours with a forecast, to at most 0.01 MWh of rounding.

- **The decomposition `err_residual_load = err_grid_load − err_renewables` is exact to within 16.5 MWh.**
  - 99.9 % of hours are within 1 MWh. The 57 exceptions all fall in 2021-01-04 .. 2021-01-06 and come entirely from the *actual*-side identity, a SMARD artefact already seen in `risk-definition.ipynb` §1.7.
  - The largest of them, 16.5 MWh, is negligible against the error sizes below.

- **An under-forecast of wind + solar shows up as an over-forecast of residual load, and the other way round.**
  - The largest residual-load error in the record, at 2026-04-05 13:00, is **−27,047 MWh**. It is almost entirely generation: wind + solar was forecast at ≈ 74,500 MWh but ≈ 44,900 MWh was recorded (`err_renewables` = +29,599 MWh), while the load error was only +2,551 MWh.
  - The actual residual load stayed just below zero all midday while the forecast expected ≈ −29,000 MWh. This is consistent with curtailment of wind and solar, which `data/smard.csv` cannot confirm.

### 2.2 Pairwise-complete hours

An hour enters a pair's metric **only if both forecast and actual exist**. Missing values are never
interpolated. The `err_*` columns are already `NaN` wherever either side is missing, so every
metric built on them follows this rule automatically. `hour_count` is always the number of hours
actually compared, per pair and per window.

Two cases produce missing hours: a forecast SMARD never published, and, after a re-fetch, the latest
forecasts whose actuals are not published yet. Both are counted from the data below, never
hardcoded.

In [ ]:
rows = []
for actual, spec in PAIRS.items():
    err = errors[ERR[actual]]
    missing = err.isna()
    missing_days = sorted({d.strftime("%Y-%m-%d") for d in err.index[missing].normalize()})
    rows.append({
        "pair": spec["label"],
        "hour_count": int(err.notna().sum()),
        "missing forecast": int(errors[spec["forecast"]].isna().sum()),
        "missing actual": int(errors[actual].isna().sum()),
        "days with a gap": ", ".join(missing_days) if missing_days else "—",
    })

pairwise = pd.DataFrame(rows).set_index("pair")

# The last hour at which all three pairs are observed: the anchor for every trailing window (§3).
JOINT_END = errors[list(ERR.values())].dropna().index.max()

print(f"rows in time_series          : {len(time_series):,}")
print(f"last jointly observed hour   : {JOINT_END}  (last row: {time_series.index.max()})")
pairwise

**Missing data is one day and affects two of the three pairs.**

- **Residual load and grid load are compared on 67,312 hours, wind + solar on all 67,336.**
  - The gap is the 24 hours of 2020-01-31, where SMARD published no grid-load forecast and therefore no residual-load forecast. Under the pairwise rule those hours drop out of those two pairs only.

- **The record currently ends on a jointly observed hour.**
  - The last row (2026-09-06 23:00) has all three forecasts and actuals, so the trailing 365-day window in §3 ends there. After a re-fetch that carries forecasts ahead of actuals, `JOINT_END` would move earlier on its own.

---

## 3 Metrics

Every number in this notebook is an average over a **window of hours**. Per pair and window:

| Metric | Definition | Unit |
|---|---|---|
| **MAE** | mean absolute error: mean of `abs(error)` | MWh |
| **RMSE** | square root of the mean of `error²` large misses are penalized more heavily than MAE. | MWh |
| **bias** | mean signed error; positive = over-forecast | MWh |
| **`hour_count`** | number of hours actually compared (pairwise-complete, §2.2) | hours |
| **nMAE** | `MAE / mean(actual)` over the same hours, so the three pairs are comparable | % |
| **capacity-normalised MAE / bias** | mean of `abs(err_renewables) / cap_wind_solar` and of `err_renewables / cap_wind_solar` — wind + solar only | % of installed capacity |

**One function computes all of them.** `window_metrics(errors, start, end, mask)` returns the
table above for any window or boolean hour mask, and `metrics_by(errors, key)` applies it per group
(month, hour of day, season, year, day).

Every table in this notebook comes from these two
functions, so all metrics share one definition. The spec numbers this helper B23 (the cascade), but
it is defined here because every slice before the cascade needs it.

**RMSE is always recomputed from the squared hourly errors of the window.** It is never averaged
across sub-windows: the mean of twelve monthly RMSEs is not the yearly RMSE.

### 3.1 Where nMAE and the capacity-normalised error are reported

**nMAE** is reported only at the **full-record, trailing-365-day, year, season and month** levels,
and as a rolling 365-day line (§4). It is **never** reported for **residual load at day level or in
level bins**. Residual load's mean over a short window can come close to zero, and the ratio
explodes there. The cell below shows how close it gets in this record. The hour-of-day slice (§4)
reports MAE and bias only.

The **capacity-normalised error** is reported for wind + solar only, at **year level** and as the
**rolling 365-day line**, and never per month. The `cap_*` columns are a yearly step value, so
within-year fleet growth is not visible, and only a full-year window matches the step. This measure
answers "did the wind + solar forecast get better?" across years: the absolute generation error grows
with the fleet even if the forecast does not get worse.

`window_metrics` enforces both rules through two switches: `nmae` (on by default) and `capacity`
(off by default).

In [ ]:
# How close does residual load's window mean get to zero? Complete days only (the §1.4 rule).
res = time_series["residual_load"]
day_obs = res.groupby(DAY).size()
daily_mean = res.groupby(DAY).mean()[day_obs >= MIN_OBS_PER_DAY]
monthly_mean = period_mean(res, "M")

print(f"residual load, whole-record mean : {res.mean():>9,.0f} MWh")
print(f"lowest monthly mean              : {monthly_mean.min():>9,.0f} MWh  ({monthly_mean.idxmin():%Y-%m})")
print(f"lowest daily mean                : {daily_mean.min():>9,.0f} MWh  ({daily_mean.idxmin():%Y-%m-%d})")
print(f"days with a mean below 5,000 MWh : {int((daily_mean < 5000).sum())} of {len(daily_mean):,}")

# Daily means against individual hours, per year: midday lows are averaged against the night.
day_min = res.groupby(DAY).min()[day_obs >= MIN_OBS_PER_DAY]
low_days = pd.DataFrame({
    "complete days": daily_mean.groupby(daily_mean.index.year).size(),
    "daily mean < 5,000 MWh": (daily_mean < 5000).groupby(daily_mean.index.year).sum(),
    "any hour < 5,000 MWh": (day_min < 5000).groupby(day_min.index.year).sum(),
    "any hour < 0 MWh": (day_min < 0).groupby(day_min.index.year).sum(),
}).rename_axis("year")
low_days

**The nMAE exclusion is needed: a daily residual-load mean can be close to zero.**

- **Monthly means are safe, daily means are not.**
  - The lowest monthly mean is 19,746 MWh (2025-06), well away from zero, so monthly nMAE stays meaningful.
  - The lowest daily mean is only **679 MWh** (2025-10-26). The full-record MAE of ≈ 2,600 MWh (§3.3) divided by that would be an nMAE of ≈ 380 %, which is a statement about the denominator, not about the forecast.
  - Only 8 of 2,806 days have a **daily mean** below 5,000 MWh, but 7 of them fall in 2025–2026
    - Individual hours go far lower: in 2026 alone, 101 days already carry at least one negative hour
    - As the solar fleet grows, more daily means will approach zero, which is why the rule is a fixed exclusion rather than a threshold.

### 3.2 `window_metrics` and `metrics_by`

- `start` and `end` are **inclusive** hour timestamps. `None` means the start or end of the record.
- `mask` is a boolean Series on `errors.index` (or an array of the same length). It selects hours inside the window.
- The result has one row per pair, indexed by the pair key (`residual_load`, `grid_load`, `renewables`). Columns that a switch turns off stay `NaN`.
- `show_metrics` only formats a table for display: pair labels, units in the column names, and columns that are empty everywhere are dropped.

In [ ]:
METRICS = ["MAE", "RMSE", "bias", "nMAE_pct", "cap_MAE_pct", "cap_bias_pct", "hour_count"]
PAIR_LABEL = {actual: spec["label"] for actual, spec in PAIRS.items()}


def window_metrics(errors, start=None, end=None, mask=None, nmae=True, capacity=False):
    """MAE, RMSE, bias and hour_count per pair over one window of hours.

    Pairwise-complete: an hour counts for a pair only where its `err_*` is not NaN. RMSE is
    computed from this window's squared hourly errors, never from sub-window RMSEs.

    nmae      include nMAE = MAE / mean(actual) in %. Callers switch it off at day level, in
              level bins and in the hour-of-day slice (Behaviour 7).
    capacity  include the capacity-normalised MAE / bias of wind + solar in % of installed
              capacity. Callers switch it on only for full-year windows (Behaviour 8).
    """
    frame = errors.loc[start:end]
    if mask is not None:
        if not isinstance(mask, pd.Series):
            mask = pd.Series(np.asarray(mask), index=errors.index)
        frame = frame[mask.reindex(frame.index, fill_value=False).astype(bool)]

    rows = {}
    for actual in PAIRS:
        err = frame[ERR[actual]]
        compared = err.notna()
        e = err[compared]
        row = dict.fromkeys(METRICS, np.nan)
        row["hour_count"] = int(compared.sum())
        if row["hour_count"]:
            row["MAE"] = e.abs().mean()
            row["RMSE"] = np.sqrt((e ** 2).mean())
            row["bias"] = e.mean()
            if nmae:
                row["nMAE_pct"] = 100 * row["MAE"] / frame.loc[compared, actual].mean()
            if capacity and actual == "renewables":
                share = e / frame.loc[compared, "cap_wind_solar"]
                row["cap_MAE_pct"] = 100 * share.abs().mean()
                row["cap_bias_pct"] = 100 * share.mean()
        rows[actual] = row

    table = pd.DataFrame.from_dict(rows, orient="index")[METRICS]
    table["hour_count"] = table["hour_count"].astype(int)
    return table.rename_axis("pair")


def metrics_by(errors, key, name=None, **switches):
    """`window_metrics` per group of `key` — the one definition, applied group by group.

    `key` is a Series on `errors.index` (or an array of the same length). Rows whose key is
    missing are left out. Returns a frame indexed by (group, pair).
    """
    name = name or getattr(key, "name", None) or "group"
    key = pd.Series(np.asarray(key), index=errors.index, name=name)
    parts = {group: window_metrics(frame, **switches)
             for group, frame in errors.groupby(key, sort=True, observed=True)}
    return pd.concat(parts, names=[name, "pair"])


DISPLAY = {
    "MAE": "MAE [MWh]", "RMSE": "RMSE [MWh]", "bias": "bias [MWh]", "nMAE_pct": "nMAE [%]",
    "cap_MAE_pct": "cap. MAE [% of cap.]", "cap_bias_pct": "cap. bias [% of cap.]",
    "hour_count": "hour_count",
}


def show_metrics(table):
    """Display formatting only: labels, units, rounding; all-empty metric columns dropped."""
    out = table.dropna(axis=1, how="all").rename(index=PAIR_LABEL)
    out = out.round({"MAE": 0, "RMSE": 0, "bias": 0,
                     "nMAE_pct": 1, "cap_MAE_pct": 2, "cap_bias_pct": 2})
    return out.rename(columns=DISPLAY)


# Sanity check against a direct computation, so the helper is not trusted blindly.
_full = window_metrics(errors)
for actual in PAIRS:
    e = errors[ERR[actual]].dropna()
    assert np.isclose(_full.loc[actual, "MAE"], e.abs().mean())
    assert np.isclose(_full.loc[actual, "RMSE"], np.sqrt((e ** 2).mean()))
    assert _full.loc[actual, "hour_count"] == len(e)
print("window_metrics matches a direct computation on the full record")

### 3.3 Headline table

Every pair over two windows:

- **full record**: every pairwise-complete hour
- **trailing 365 days**: the 365 days ending at `JOINT_END`, the last hour at which all three pairs are observed (§2.2), derived from the data

These are the provisional benchmark numbers. The modelling spec re-scores SMARD on its own test
window from the hourly export (§8).

In [ ]:
TRAIL_START = JOINT_END - WINDOW + RESOLUTION   # exactly 365 days of hours, ending at JOINT_END

headline = pd.concat(
    {
        "full record": window_metrics(errors),
        "trailing 365 days": window_metrics(errors, start=TRAIL_START, end=JOINT_END),
    },
    names=["window"],
).swaplevel().reindex(pd.MultiIndex.from_product(
    [list(PAIRS), ["full record", "trailing 365 days"]], names=["pair", "window"]
))

print(f"full record      : {errors.index.min()} .. {errors.index.max()}")
print(f"trailing 365 days: {TRAIL_START} .. {JOINT_END}")
show_metrics(headline)

**SMARD misses residual load by ≈ 2,600–2,800 MWh per hour on average. The recent year is slightly worse and has flipped from under- to over-forecasting.**

- **Residual load: MAE 2,584 MWh over the full record, 2,790 MWh over the trailing 365 days.**
  - nMAE rises from 7.8 % to 10.1 %. Part of that is the smaller denominator (residual load keeps falling as wind + solar grow), not only a larger error. §4 separates the two with rolling lines.
  - RMSE (3,349 / 3,647 MWh) is ≈ 1.3× the MAE in both windows, so a few large misses carry noticeable weight.

- **The residual-load bias changed sign: −402 MWh (full record) → +414 MWh (trailing 365 days).**
  - The bias decomposes exactly (`bias_res = bias_load − bias_gen`): full record −494 − (−93) = −401; trailing 316 − (−98) = +414.
  - So the flip comes from **grid load**, which moved from under-forecast (−494 MWh) to over-forecast (+316 MWh). The wind + solar bias hardly moved (≈ −95 MWh in both windows).

- **Relative to its size, grid load is the most accurate pair.**
  - Grid load nMAE is 3.8 % in both windows. Wind + solar is at ≈ 7 %.
  - The wind + solar MAE grew from 1,505 to 1,782 MWh, but its nMAE stayed flat (7.0 % → 6.9 %). The larger absolute error tracks the larger generation, not a worse forecast. §4 checks this against installed capacity.

- **The load and generation errors are close to uncorrelated.**
  - Residual-load RMSE² ≈ load RMSE² + generation RMSE²: √(2,638² + 2,074²) ≈ 3,356 MWh against the measured 3,349 MWh over the full record, and 3,630 against 3,647 MWh over the trailing year. That only holds if the two component errors barely co-move. §5 reports the correlation per level bin.

- **`hour_count` is 8,759 for the trailing window, not 8,760:** 365 days × 24 h minus the missing 02:00 of the spring DST switch.